In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, glob, lmdb, pickle, argparse
import numpy as np

def as_energy(obj):
    """
    从样本对象里稳健地取出能量数值（float）。
    兼容 attribute / dict 两种风格，以及 torch/tensor -> float。
    """
    val = None
    # 优先 energy，再退回 y
    for key in ("energy", "y"):
        # 先试 attribute
        if hasattr(obj, key):
            val = getattr(obj, key)
            break
        # 再试 dict-like
        if hasattr(obj, "get"):
            v = obj.get(key, None)
            if v is not None:
                val = v
                break
    if val is None:
        raise KeyError("sample does not contain energy or y field")
    try:
        # 兼容 numpy / torch tensor
        return float(val) if np.isscalar(val) else float(np.array(val).reshape(()))
    except Exception:
        # 最后兜底
        return float(val)

def as_fid(obj):
    """
    取出 fid（int）。作者制作 LMDB 时，最优样本 fid = -1。
    """
    if hasattr(obj, "fid"):
        return int(getattr(obj, "fid"))
    if hasattr(obj, "get"):
        v = obj.get("fid", None)
        if v is not None:
            return int(v)
    raise KeyError("sample does not contain fid field")

def read_length(txn):
    """
    读取分片里的样本总数：作者写入了 'length'。
    若没有，就线性探测（不推荐，但做兜底）。
    """
    raw = txn.get(b"length")
    if raw is not None:
        try:
            return int(pickle.loads(raw))
        except Exception:
            pass
    # 兜底线性探测（慢）：从 0 开始直到 miss 连续 N 次（N=100）
    print("WARNING: 'length' key not found, falling back to slow probing...")
    miss = 0
    i = 0
    while miss < 100:  # 简单兜底
        if txn.get(str(i).encode("ascii")) is None:
            miss += 1
        else:
            miss = 0
        i += 1
    return max(0, i - miss)

def scan_lmdb(db_path, atol):
    total = 0
    n_fid_neg1 = 0
    bad = []  # 存储 (db_path, key_str, energy) 违反条件的样本
    with lmdb.open(db_path, subdir=False, readonly=True, lock=False,
                   readahead=True, meminit=False, max_readers=1) as env:
        with env.begin() as txn:
            length = read_length(txn)
            total = length
            for i in range(length):
                key = str(i).encode("ascii")
                raw = txn.get(key)
                if raw is None:
                    # 有些分片可能存在空洞，跳过
                    continue
                obj = pickle.loads(raw)
                try:
                    fid = as_fid(obj)
                except KeyError:
                    # 没有 fid 的样本跳过
                    continue
                if fid == -1:
                    n_fid_neg1 += 1
                    try:
                        e = as_energy(obj)
                    except KeyError:
                        bad.append((db_path, key.decode(), "NO_ENERGY_FIELD"))
                        continue
                    if not np.isclose(e, 0.0, atol=atol):
                        bad.append((db_path, key.decode(), e))
    return total, n_fid_neg1, bad

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--lmdb_dir", default="train_allE", help="包含若干 *.lmdb 分片的目录")
    ap.add_argument("--pattern", default="*.lmdb", help="分片文件匹配模式，默认 *.lmdb")
    ap.add_argument("--atol", type=float, default=1e-8, help="判断能量等于 0 的绝对容差")
    args = ap.parse_args()

    lmdb_files = sorted(glob.glob(os.path.join(args.lmdb_dir, args.pattern)))
    if not lmdb_files:
        print(f"[ERROR] 没在 {args.lmdb_dir} 下找到匹配 {args.pattern} 的 lmdb 文件")
        return

    grand_total = 0
    grand_fid_neg1 = 0
    all_bad = []

    print(f"扫描 {len(lmdb_files)} 个分片...\n")
    for path in lmdb_files:
        t, nneg1, bad = scan_lmdb(path, args.atol)
        grand_total += t
        grand_fid_neg1 += nneg1
        all_bad.extend(bad)
        print(f"- {os.path.basename(path)}: 样本 {t:,} 条，fid==-1 记录 {nneg1:,} 条，异常 {len(bad)} 条")

    print("\n====== 汇总 ======")
    print(f"总样本数：{grand_total:,}")
    print(f"fid == -1 的记录总数：{grand_fid_neg1:,}")
    if all_bad:
        print(f"❌ 有 {len(all_bad)} 条 fid==-1 的记录能量不为 0（或缺失能量字段，容差 atol={args.atol}）")
        # 列出前若干条异常，防止刷屏
        preview = 50
        print(f"以下列出前 {min(preview, len(all_bad))} 条：")
        for (db, key, e) in all_bad[:preview]:
            print(f"  - {os.path.basename(db)} [key={key}]  energy={e}")
        if len(all_bad) > preview:
            print(f"... 其余 {len(all_bad) - preview} 条已省略")
    else:
        print("✅ 所有 fid==-1 的记录能量均为 0（在给定容差内）")

if __name__ == "__main__":
    main()


Data(pos=[50, 3], cell=[1, 3, 3], atomic_numbers=[50], natoms=50, tags=[50], edge_index=[2, 990], cell_offsets=[990, 3], fixed=[50], sid='74_122_9', fid=0, y=0.02296985999998924)
['tags', 'fixed', 'sid', 'cell', 'natoms', 'pos', 'y', 'edge_index', 'cell_offsets', 'atomic_numbers', 'fid']
pos <class 'torch.Tensor'> torch.Size([50, 3])
cell <class 'torch.Tensor'> torch.Size([1, 3, 3])
atomic_numbers <class 'torch.Tensor'> torch.Size([50])
natoms <class 'int'> None
tags <class 'torch.Tensor'> torch.Size([50])
edge_index <class 'torch.Tensor'> torch.Size([2, 990])
cell_offsets <class 'torch.Tensor'> torch.Size([990, 3])
fixed <class 'torch.Tensor'> torch.Size([50])
sid <class 'str'> None
fid <class 'int'> None
y <class 'torch.Tensor'> torch.Size([])
tensor([[ 1.5792,  6.3887, 18.5132],
        [ 1.5576,  3.0676, 15.1903],
        [ 1.6008,  9.7098, 21.8361],
        [ 4.8249,  6.8164, 18.0646],
        [ 4.8033,  3.4953, 14.7417],
        [ 4.7817,  0.1742, 18.0611],
        [ 1.7728,  6.4

In [1]:
from adsorbdiff.utils.utils import build_config
from types import SimpleNamespace
cfg = build_config(SimpleNamespace(mode='train', config_yml='/root/autodl-tmp/AdsorbFlow/configs/flow/painn_so3_flow.yml')
, override_args=[])
print("EFFECTIVE imports =", cfg.get('imports'))
print("EFFECTIVE trainer =", cfg.get('trainer'))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/root/miniconda3/envs/adsorbdiff/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in l

TypeError: build_config() got an unexpected keyword argument 'override_args'

In [20]:
from adsorbdiff.datasets import LmdbDataset
ds = LmdbDataset({"src": "/root/autodl-tmp/AdsorbFlow/0/final_struct_lmdb"})
for d in ds:
      print(d.atomic_numbers)


tensor([79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79.,
        79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79., 79.,
        79., 79., 79., 79., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 16., 16., 16., 16., 16.,  7.,  7.,  8.])
tensor([22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22.,
        22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22.,
        22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 22., 33., 33.,
        33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33.,
        33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33., 33.,
        33., 33., 33., 33., 33., 33., 33., 33., 33., 33.,  6.,  1.,  1.,  1.,
         8.])
tensor([72., 72., 72., 72., 72., 72., 72., 72., 72., 72., 72., 72., 72., 72.,
        72., 72., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78.,
        78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 78., 7

In [18]:
import os

print(os.getcwd())  # 当前工作目录
print(os.listdir("/root/autodl-tmp/AdsorbFlow/"))  # 看看目录里到底有哪些文件夹

/root/autodl-tmp/AdsorbFlow
['.git', '.gitignore', 'LICENSE', 'README.md', 'adsorbdiff', 'configs', 'examples', 'licenses', 'main.py', 'requirements.txt', 'run.py', 'scripts', 'setup.py', 'adsorbdiff.egg-info', 'submit.sh', 'test.ipynb', 'val_nonrelaxed_update']


In [17]:
import pickle

# 修改为你的文件路径
file_path = "/root/autodl-tmp/AdsorbFlow/oc20_dense_mappings/oc20dense_compute.pkl"

# 打开并加载 pkl 文件
with open(file_path, "rb") as f:
    data = pickle.load(f)

# 打印对象类型
print("Loaded object type:", type(data))

# 打印前几个内容（根据对象类型不同显示方式也不同）
if isinstance(data, dict):
    print("Keys:", len(data.keys()))
if isinstance(data, dict):
    print("Keys:", list(data.keys()))
elif isinstance(data, (list, tuple)):
    print("Length:", len(data))
    print("First 5 elements:", data[:5])
else:
    print("Content preview:", str(data)[:500])  # 打印前 500 个字符
print(data['0_1190_0'])


Loaded object type: <class 'collections.defaultdict'>
Keys: 973
Keys: ['0_1190_0', '0_11190_86', '0_2302_72', '0_2336_45', '0_2374_49', '0_2554_20', '0_3161_13', '0_3572_18', '0_3577_7', '0_5107_96', '0_5576_5', '0_5969_37', '0_6055_19', '0_8478_44', '10_133_5', '10_1444_17', '10_394_10', '10_3099_33', '10_3099_5', '10_3353_5', '10_4609_10', '10_5863_121', '11_1573_56', '11_1728_26', '11_2154_129', '11_2773_23', '11_783_2', '12_1174_85', '12_1757_5', '12_1990_4', '12_2492_47', '12_3701_25', '12_3915_5', '12_6446_39', '12_7178_20', '12_8736_30', '12_1097_22', '13_1974_3', '13_2561_4', '13_3033_3', '13_34_2', '13_706_25', '13_734_133', '14_1662_2', '14_5218_40', '14_6259_6', '15_1204_18', '15_159_1', '15_3994_47', '15_1267_21', '15_3890_77', '15_9991_19', '16_3177_11', '16_4103_115', '16_4335_2', '16_5258_75', '16_6384_9', '16_834_196', '16_7283_37', '16_953_8', '17_2359_31', '17_2568_4', '17_2796_98', '17_405_22', '17_3066_19', '17_3068_40', '17_3555_20', '17_6587_7', '17_8286_22', '17_